# EVODEX + ReactionT5 Evaluation (Bm/Cm/Dm/Em and B/C/D/E) — v5

This notebook runs a molecular transformer (ReactionT5) to generate predicted
products for a set of substrates, then evaluates each prediction against the
EVODEX abstractions:

- Mechanistic / partial operators: Bm, Cm, Dm, Em (via `mechanistic_labeler`).
- Complete operators: B, C, D, E (via `evaluation.find_exact_matching_operators`).

For each substrate we:

1. Define the substrate SMILES (and optional reagents).
2. Use the ReactionT5 model to generate 50 candidate products and scores.
3. Run Bm/Cm/Dm/Em and B/C/D/E on every prediction.
4. Build a combined CSV of all predictions, with 8 columns of operator IDs:
   - `Bm_match_id`, `Cm_match_id`, `Dm_match_id`, `Em_match_id`
   - `B_match`, `C_match`, `D_match`, `E_match`
5. Generate one structures PDF panel per substrate:
   - top-left cell: substrate (boxed, legend "substrate");
   - first 15 predictions by rank;
   - legends under products show only the rank;
   - cells get nested color boxes indicating which abstractions hit;
   - when available, we draw the mechanistically mapped product SMILES.

We also write a third CSV with metadata for all operators that were hit, pulled
from the EVODEX CSVs.

## 1. Environment setup

Install RDKit and the transformer stack, clone the `evodex.2` branch of the
EVODEX repo, and add it to `sys.path`.

In [11]:
!pip install rdkit transformers sentencepiece -q
!pip install cairosvg

# Clone EVODEX (evodex.2 branch, which includes mechanistic_labeler)
!git clone -b evodex.2 https://github.com/UCB-BioE-Anderson-Lab/EVODEX.git

import sys, os

REPO_ROOT = "/content/EVODEX"
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

print("Repo cloned to:", REPO_ROOT)
print("Contents:", os.listdir(REPO_ROOT))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 5.4 MB/s eta 0:00:00
fatal: destination path 'EVODEX' already exists and is not an empty directory.
Repo cloned to: /content/EVODEX
Contents: ['evodex_icon.png', 'website', 'evodex', 'README.md', 'notebooks', '.gitignore', 'evodex_logo.ai', 'pipeline', 'MANIFEST.in', 'LICENSE', 'tests', 'run_pipeline.py', 'analysis_reports', 'analysis', 'setup.py', '.git', 'index.html', 'requirements.txt']


## 2. Imports

- RDKit (core + drawing)
- Pandas and Matplotlib
- HuggingFace `transformers` for ReactionT5
- EVODEX evaluation utilities

In [12]:
from typing import Dict, Any, Optional, List, Tuple

import pandas as pd
import matplotlib.pyplot as plt

from rdkit import Chem
from rdkit.Chem import Draw
from rdkit import RDLogger

RDLogger.DisableLog("rdApp.*")

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

from evodex import evaluation
from evodex.mechanistic_labeler import (
    find_mechanistic_match_in_dataset,
    prepare_reaction,
    mechanistic_label_reaction,
)

print("Imports successful.")

Imports successful.


## 3. Configure the molecular transformer (ReactionT5)

We use the forward ReactionT5 model from HuggingFace:

- Model: `sagawa/ReactionT5v2-forward`
- Interface: `predict_products(reactants, reagents, num_beams, num_return_sequences)`

Each call returns a list of `(product_smiles, score)` pairs.

In [13]:
MODEL_NAME = "sagawa/ReactionT5v2-forward"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("Loaded model:", MODEL_NAME, "on device:", device)

def predict_products(
    reactants: str,
    reagents: str = "",
    num_beams: int = 50,
    num_return_sequences: int = 50,
):
    # Run ReactionT5 and return a list of (product_smiles, score).
    prompt = f"REACTANT:{reactants}REAGENT:{reagents}"
    enc = tokenizer(prompt, return_tensors="pt")
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        out = model.generate(
            **enc,
            num_beams=num_beams,
            num_return_sequences=num_return_sequences,
            return_dict_in_generate=True,
            output_scores=True,
        )

    seqs = out.sequences
    seq_scores = out.sequences_scores

    preds = []
    for seq, score in zip(seqs, seq_scores):
        text = tokenizer.decode(seq, skip_special_tokens=True)
        smiles = text.replace(" ", "").rstrip(".")
        preds.append((smiles, float(score)))
    return preds

# Quick smoke test (small beams) on propanol
test_preds = predict_products("CCCO", num_beams=4, num_return_sequences=4)
for i, (p, s) in enumerate(test_preds, start=1):
    print(f"{i}. {p}   score={s:.3f}")

Loaded model: sagawa/ReactionT5v2-forward on device: cpu
1. CCCOC(=O)CCC   score=-0.034
2. CCCOC(=O)Cc1ccccc1   score=-0.120
3. CCCOC(=O)CC1CO1   score=-0.396
4. C=CC1CO1   score=-0.453


## 4. Define substrates to evaluate

Edit the list below to control which substrates are evaluated. Each entry may
optionally include a `reagents` field.

Example entries:

- `{"substrate_name": "propanol", "substrate_smiles": "CCCO", "reagents": ""}`
- `{"substrate_name": "ethyl acetate", "substrate_smiles": "CCOC(=O)C", "reagents": ""}`

In [39]:
# Edit this list as needed.
substrates = [
    {
        "substrate_name": "propanol",
        "substrate_smiles": "CCCO",
        "reagents": "",
    },
    {
        "substrate_name": "ethyl acetate",
        "substrate_smiles": "CCOC(=O)C",
        "reagents": "",
    },
    {
        "substrate_name": "propene",
        "substrate_smiles": "CC=C",
        "reagents": "",
    },
    # Add more substrates here as needed
]

# Generation settings for all substrates
NUM_BEAMS = 50
NUM_RETURN_SEQUENCES = 50

substrates

[{'substrate_name': 'propanol', 'substrate_smiles': 'CCCO', 'reagents': ''},
 {'substrate_name': 'ethyl acetate',
  'substrate_smiles': 'CCOC(=O)C',
  'reagents': ''},
 {'substrate_name': 'propene', 'substrate_smiles': 'CC=C', 'reagents': ''}]

## 5. Generate transformer predictions

For each substrate, we call `predict_products` and build a combined
`predictions_df` with columns:

- `substrate_name`
- `substrate_smiles`
- `reagents`
- `rank`
- `model_score`
- `predicted_product`

We keep all 50 predictions per substrate in the CSV; the structure panels
later render only the first 15 per substrate.

In [40]:
records: List[Dict[str, Any]] = []

for sub in substrates:
    name = sub["substrate_name"]
    smi = sub["substrate_smiles"]
    reag = sub.get("reagents", "")

    print(f"Predicting for {name} ({smi})...")
    preds = predict_products(
        reactants=smi,
        reagents=reag,
        num_beams=NUM_BEAMS,
        num_return_sequences=NUM_RETURN_SEQUENCES,
    )

    for rank, (pdt, score) in enumerate(preds, start=1):
        records.append(
            {
                "substrate_name": name,
                "substrate_smiles": smi,
                "reagents": reag,
                "rank": rank,
                "model_score": score,
                "predicted_product": pdt,
            }
        )

predictions_df = pd.DataFrame.from_records(records)
print("Total predictions:", len(predictions_df))
predictions_df.head()

Predicting for propanol (CCCO)...
Predicting for ethyl acetate (CCOC(=O)C)...
Predicting for propene (CC=C)...
Total predictions: 150


,substrate_name,substrate_smiles,reagents,rank,model_score,predicted_product
0,propanol,CCCO,,1,-0.034429,CCCOC(=O)CCC
1,propanol,CCCO,,2,-0.120405,CCCOC(=O)Cc1ccccc1
2,propanol,CCCO,,3,-0.313646,CCCOC(=O)[C@@H]1CCC[C@@H]1C(=O)OCCC
3,propanol,CCCO,,4,-0.320872,OC[C@@H]1CC[C@H]2CN3CCC4CC5CC(=O)CC[C@]5(C)[C@...
4,propanol,CCCO,,5,-0.336907,OC[C@@H]1CC[C@H]2CN3CCC4CC5CC(=O)CC[C@]5(C)[C@...


## 6. EVODEX evaluation helpers (Bm/Cm/Dm/Em and B/C/D/E)

These helpers:

- Build `substrate>>product` reaction SMILES.
- For mechanistic abstractions (Bm, Cm, Dm, Em) call
  `find_mechanistic_match_in_dataset(..., abstraction)`.
- For complete abstractions (B, C, D, E) call
  `evaluation.find_exact_matching_operators(..., evodex_type=...)`.

We return:

- `*_match_id` columns for Bm, Cm, Dm, Em (single IDs or `no_match` / `invalid_smiles`).
- `*_match` columns for B, C, D, E (semicolon-separated IDs or `no_match` / `invalid_smiles`).

We also build a `mapped_product_for_panel` column per row, choosing the best
mechanistic mapping available (Em first, then Dm, Cm, Bm) for use in structure
panels so atom maps are visible.

In [41]:
MECH_LEVELS = ["Bm", "Cm", "Dm", "Em"]
COMP_LEVELS = ["B", "C", "D", "E"]

NON_HITS = {"no_match", "invalid_smiles"}


def _safe_mol_from_smiles(smi: str) -> Optional[Chem.Mol]:
    if not isinstance(smi, str) or not smi.strip():
        return None
    try:
        mol = Chem.MolFromSmiles(smi)
    except Exception:
        return None
    return mol


def _mapped_product_smiles_from_labeling(labeling: Dict[str, Any]) -> Optional[str]:
    # Construct mapped product SMILES from a mechanistic_label_reaction result.
    try:
        rxn = labeling["rxn"]
        prod = rxn.GetProducts()[0]
    except Exception:
        return None

    prod = Chem.Mol(prod)
    mapped_r, mapped_p = labeling.get("mapped_atoms", ([], []))

    for atom_idx, map_num in mapped_p:
        try:
            prod.GetAtomWithIdx(atom_idx).SetAtomMapNum(int(map_num))
        except Exception:
            pass

    try:
        smi = Chem.MolToSmiles(prod)
    except Exception:
        return None
    return smi


def run_mech_match(level: str, substrate_smiles: str, product_smiles: str) -> Dict[str, Any]:
    # Mechanistic match for one level (Bm/Cm/Dm/Em).
    col_id = f"{level}_match_id"
    col_mapped = f"{level}_mapped_product"

    if _safe_mol_from_smiles(substrate_smiles) is None or _safe_mol_from_smiles(product_smiles) is None:
        return {col_id: "invalid_smiles", col_mapped: None}

    rxn_smiles = f"{substrate_smiles}>>{product_smiles}"
    try:
        match = find_mechanistic_match_in_dataset(rxn_smiles, level)
    except Exception:
        return {col_id: "invalid_smiles", col_mapped: None}

    if match is None:
        return {col_id: "no_match", col_mapped: None}

    labeling = match.get("labeling", {})
    mapped_prod = _mapped_product_smiles_from_labeling(labeling)

    return {
        col_id: match.get("dataset_id", "unknown"),
        col_mapped: mapped_prod,
    }


def run_complete_match(level: str, substrate_smiles: str, product_smiles: str) -> Dict[str, Any]:
    # Complete-operator match for one level (B/C/D/E).
    col = f"{level}_match"

    if _safe_mol_from_smiles(substrate_smiles) is None or _safe_mol_from_smiles(product_smiles) is None:
        return {col: "invalid_smiles"}

    rxn_smiles = f"{substrate_smiles}>>{product_smiles}"
    try:
        ids: List[str] = evaluation.find_exact_matching_operators(
            rxn_smiles, evodex_type=level
        )
    except Exception:
        return {col: "invalid_smiles"}

    if not ids:
        return {col: "no_match"}

    return {col: ";".join(ids)}

## 7. Run EVODEX evaluation on all predictions

We annotate each prediction with:

- `Bm_match_id`, `Cm_match_id`, `Dm_match_id`, `Em_match_id`
- `Bm_mapped_product`, `Cm_mapped_product`, `Dm_mapped_product`, `Em_mapped_product`
- `B_match`, `C_match`, `D_match`, `E_match`
- `mapped_product_for_panel` (best available mapped product for drawing).

We then build:

- `full_df`: combined table for all substrates.
- `tables_by_substrate`: dict of per-substrate tables.

In [42]:
def evaluate_predictions(
    predictions: pd.DataFrame,
) -> Tuple[pd.DataFrame, Dict[str, pd.DataFrame]]:
    records: List[Dict[str, Any]] = []

    required_cols = {"substrate_name", "substrate_smiles", "rank", "model_score", "predicted_product"}
    missing = required_cols - set(predictions.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    for _, row in predictions.iterrows():
        sub_name = row["substrate_name"]
        sub_smiles = row["substrate_smiles"]
        rank = row["rank"]
        score = row["model_score"]
        pdt_smiles = row["predicted_product"]

        rec: Dict[str, Any] = {
            "substrate_name": sub_name,
            "substrate_smiles": sub_smiles,
            "rank": rank,
            "model_score": score,
            "predicted_product": pdt_smiles,
        }

        # Mechanistic
        for lvl in MECH_LEVELS:
            mech_res = run_mech_match(lvl, sub_smiles, pdt_smiles)
            rec.update(mech_res)

        # Complete operators
        for lvl in COMP_LEVELS:
            comp_res = run_complete_match(lvl, sub_smiles, pdt_smiles)
            rec.update(comp_res)

        # Choose best mapped product for panel (prioritize finer abstractions)
        mapped_product_for_panel = None
        priority = ["Em", "Dm", "Cm", "Bm"]
        for lvl in priority:
            val = rec.get(f"{lvl}_mapped_product")
            if isinstance(val, str):
                mapped_product_for_panel = val
                break

        rec["mapped_product_for_panel"] = mapped_product_for_panel

        records.append(rec)

    full_df = pd.DataFrame.from_records(records)

    tables_by_substrate: Dict[str, pd.DataFrame] = {}
    for sub_name, group in full_df.groupby("substrate_name", sort=False):
        df = group.sort_values("rank").reset_index(drop=True)
        tables_by_substrate[sub_name] = df

    return full_df, tables_by_substrate


full_df, tables_by_substrate = evaluate_predictions(predictions_df)

print("Substrates:", list(tables_by_substrate.keys()))
full_df.head()

Substrates: ['propanol', 'ethyl acetate', 'propene']


,substrate_name,substrate_smiles,rank,model_score,predicted_product,Bm_match_id,Bm_mapped_product,Cm_match_id,Cm_mapped_product,Dm_match_id,Dm_mapped_product,Em_match_id,Em_mapped_product,B_match,C_match,D_match,E_match,mapped_product_for_panel
0,propanol,CCCO,1,-0.034429,CCCOC(=O)CCC,EVODEX.2-Bm94,[H]C([H])([H])C([H])([H])C([H])([H])C(=O)[O:60...,EVODEX.2-Cm276,[H]C([H])([H])C([H])([H])C([H])([H])C(=O)[O:23...,EVODEX.2-Dm387,[H]C([H])([H])C([H])([H])C([H])([H])C(=O)[O:27...,EVODEX.2-Em410,[H]C([H])([H])C([H])([H])C([H])([H])C(=O)[O:27...,no_match,no_match,no_match,no_match,[H]C([H])([H])C([H])([H])C([H])([H])C(=O)[O:27...
1,propanol,CCCO,2,-0.120405,CCCOC(=O)Cc1ccccc1,EVODEX.2-Bm94,[H]c1c([H])c([H])c(C([H])([H])C(=O)[O:60]C([H]...,EVODEX.2-Cm276,[H]c1c([H])c([H])c(C([H])([H])C(=O)[O:23][C:22...,EVODEX.2-Dm387,[H]c1c([H])c([H])c(C([H])([H])C(=O)[O:27][C:26...,EVODEX.2-Em410,[H]c1c([H])c([H])c(C([H])([H])C(=O)[O:27][C:26...,no_match,no_match,no_match,no_match,[H]c1c([H])c([H])c(C([H])([H])C(=O)[O:27][C:26...
2,propanol,CCCO,3,-0.313646,CCCOC(=O)[C@@H]1CCC[C@@H]1C(=O)OCCC,EVODEX.2-Bm94,[H]C([H])([H])C([H])([H])C([H])([H])OC(=O)[C@@...,EVODEX.2-Cm276,[H]C([H])([H])C([H])([H])C([H])([H])OC(=O)[C@@...,EVODEX.2-Dm387,[H]C([H])([H])C([H])([H])C([H])([H])OC(=O)[C@@...,EVODEX.2-Em410,[H]C([H])([H])C([H])([H])C([H])([H])OC(=O)[C@@...,no_match,no_match,no_match,no_match,[H]C([H])([H])C([H])([H])C([H])([H])OC(=O)[C@@...
3,propanol,CCCO,4,-0.320872,OC[C@@H]1CC[C@H]2CN3CCC4CC5CC(=O)CC[C@]5(C)[C@...,EVODEX.2-Bm178,[H]OC([H])([H])[C@]1([H])C([H])([H])C([H])([H]...,EVODEX.2-Cm554,[H]OC([H])([H])[C@]1([H])C([H])([H])[C:10]([H:...,EVODEX.2-Dm641,[H]OC([H])([H])[C@]1([H])C([H])([H])[C:10]([H:...,EVODEX.2-Em648,[H]OC([H])([H])[C@]1([H])C([H])([H])[C:10]([H:...,no_match,no_match,no_match,no_match,[H]OC([H])([H])[C@]1([H])C([H])([H])[C:10]([H:...
4,propanol,CCCO,5,-0.336907,OC[C@@H]1CC[C@H]2CN3CCC4CC5CC(=O)CC[C@]5(C)[C@...,invalid_smiles,None,invalid_smiles,None,invalid_smiles,None,invalid_smiles,None,no_match,no_match,no_match,no_match,None


## 8. Save combined CSV and operator metadata CSV

We write:

1. A combined CSV with all predictions and 8 operator columns:
   - `Bm_match_id`, `Cm_match_id`, `Dm_match_id`, `Em_match_id`
   - `B_match`, `C_match`, `D_match`, `E_match`

2. A CSV summarizing all operators that were hit, with their metadata
   from the EVODEX CSVs and a `hit_count` column:

   - `evodex_type` (Bm/Cm/Dm/Em/B/C/D/E)
   - `id`, `smirks`, `sources`, `hash`
   - `hit_count`

In [43]:
combined_csv_path = "evodex_transformer_evodex_evaluation_table.csv"
full_df.to_csv(combined_csv_path, index=False)
print("Wrote combined table to:", combined_csv_path)

# Build operator metadata table
data_dir = os.path.join(REPO_ROOT, "evodex", "data")
DATA_FILES = {
    "Bm": "EVODEX-Bm.csv",
    "Cm": "EVODEX-Cm.csv",
    "Dm": "EVODEX-Dm.csv",
    "Em": "EVODEX-Em.csv",
    "B": "EVODEX-B.csv",
    "C": "EVODEX-C.csv",
    "D": "EVODEX-D.csv",
    "E": "EVODEX-E.csv",
}

operator_hits: List[Dict[str, Any]] = []

# Mechanistic hits
for lvl in MECH_LEVELS:
    col = f"{lvl}_match_id"
    if col not in full_df.columns:
        continue
    series = full_df[col]
    valid = series[~series.isin(NON_HITS)]
    counts = valid.value_counts()
    for op_id, cnt in counts.items():
        operator_hits.append(
            {"evodex_type": lvl, "id": op_id, "hit_count": int(cnt)}
        )

# Complete hits
for lvl in COMP_LEVELS:
    col = f"{lvl}_match"
    if col not in full_df.columns:
        continue
    series = full_df[col]
    valid = series[~series.isin(NON_HITS)]
    if valid.empty:
        continue
    exploded = valid.str.split(";").explode().str.strip()
    counts = exploded.value_counts()
    for op_id, cnt in counts.items():
        operator_hits.append(
            {"evodex_type": lvl, "id": op_id, "hit_count": int(cnt)}
        )

if operator_hits:
    hits_df = pd.DataFrame(operator_hits)
    frames = []
    for lvl in sorted(hits_df["evodex_type"].unique(), key=lambda x: (x.endswith("m"), x)):
        df_lvl = hits_df[hits_df["evodex_type"] == lvl]
        csv_name = DATA_FILES.get(lvl)
        if csv_name is None:
            continue
        path = os.path.join(data_dir, csv_name)
        meta = pd.read_csv(path)
        merged = df_lvl.merge(meta, on="id", how="left")
        frames.append(merged)
    operator_meta_df = pd.concat(frames, ignore_index=True)
else:
    operator_meta_df = pd.DataFrame(columns=["evodex_type", "id", "hit_count", "smirks", "sources", "hash"])

operator_meta_path = "evodex_operator_hits_metadata.csv"
operator_meta_df.to_csv(operator_meta_path, index=False)
print("Wrote operator metadata to:", operator_meta_path)

operator_meta_df.head()

Wrote combined table to: evodex_transformer_evodex_evaluation_table.csv
Wrote operator metadata to: evodex_operator_hits_metadata.csv


,evodex_type,id,hit_count,smirks,sources,hash
0,Bm,EVODEX.2-Bm178,16,[#1]-[#6@:63]>>[#6]-[#6@@:63],"EVODEX.2-P1634,EVODEX.2-P1644,EVODEX.2-P1661,E...",9572123006982728315883921443164416594800943999...
1,Bm,EVODEX.2-Bm94,5,[#8:60]-[#1]>>[#6]-[#8:60],"EVODEX.2-P1367,EVODEX.2-P1370,EVODEX.2-P1382,E...",9056836375910106651849524092346340331280237551...
2,Cm,EVODEX.2-Cm276,5,[#8:23](-[#6@:22])-[#1]>>[#8:23](-[#6@:22])-[#...,"EVODEX.2-P1573,EVODEX.2-P1576,EVODEX.2-P1580,E...",6083435755351300467612855443790021556232773008...
3,Cm,EVODEX.2-Cm940,5,[#6:1](-[#6:2])(-[#1:14])(-[#1:15])-[#1]>>[#6:...,"EVODEX.2-P5582,EVODEX.2-P5583,EVODEX.2-P5588,E...",3262159593727522969518703619382879018360029949...
4,Cm,EVODEX.2-Cm554,4,[#6:10](-[#6@:11])(-[#1:16])(-[#1:17])-[#1]>>[...,"EVODEX.2-P2850,EVODEX.2-P2888,EVODEX.2-P3005,E...",6946420363978032040223458631314576028599004629...


## 9. Structure panels (multi-page PDF)

We build one structure panel per substrate:

- A 4×4 grid of cells.
- Top-left cell is the substrate, legend "substrate", with a blue box.
- The remaining cells are the first 15 predictions by rank (one per cell).
- Each product cell legend is just the rank number.
- Cells get nested colored rectangles indicating which abstractions hit:

  - Mechanistic abstractions (outer to inner):
    - Bm: dark green
    - Cm: purple
    - Dm: orange
    - Em: red
  - Complete abstractions (further inner rectangles):
    - B: dark blue
    - C: teal
    - D: brown
    - E: black

- For each product, if `mapped_product_for_panel` parses, we draw that; otherwise we
  fall back to the raw predicted product.

The panels are saved to:

- per-substrate PNGs
- a single multi-page PDF: `evodex_structure_panels.pdf`

In [44]:
# === 9. Structure panels (SVG) ===

import os
from typing import Optional, List
from rdkit.Chem import Draw

os.makedirs("evodex_structure_panels", exist_ok=True)

# Grid config
MOLS_PER_ROW = 4
N_ROWS = 4                        # 4x4 grid
SUB_IMG_SIZE = (250, 250)
N_PANEL = MOLS_PER_ROW * N_ROWS   # 1 substrate + up to 15 products


def make_structure_panel_svg(sub_name: str, df: pd.DataFrame) -> str:
    # Substrate mol in (0,0)
    sub_smiles = df["substrate_smiles"].iloc[0]
    sub_mol = _safe_mol_from_smiles(sub_smiles)

    mols: List[Optional[Chem.Mol]] = [sub_mol]
    legends: List[str] = ["substrate"]

    # First 15 predictions by rank
    df_panel = df.sort_values("rank").head(N_PANEL - 1)

    for _, row in df_panel.iterrows():
        pdt = row["predicted_product"]
        mapped = row.get("mapped_product_for_panel")
        rank = row["rank"]

        mol = None
        if isinstance(mapped, str):
            mol = _safe_mol_from_smiles(mapped)
        if mol is None:
            mol = _safe_mol_from_smiles(pdt)

        mols.append(mol)
        legends.append(str(int(rank)))

    # Pad to full grid
    while len(mols) < N_PANEL:
        mols.append(None)
        legends.append("")

    # Base SVG from RDKit
    svg_obj = Draw.MolsToGridImage(
        mols,
        molsPerRow=MOLS_PER_ROW,
        legends=legends,
        subImgSize=SUB_IMG_SIZE,
        useSVG=True,
    )

    # --- Normalize to SVG text ---
    if hasattr(svg_obj, "GetDrawingText"):        # RDKit SVG drawer object
        svg = svg_obj.GetDrawingText()
    elif hasattr(svg_obj, "data"):                # IPython.display.SVG object
        data = svg_obj.data
        if isinstance(data, bytes):
            svg = data.decode("utf-8")
        else:
            svg = str(data)
    elif isinstance(svg_obj, bytes):              # Raw bytes
        svg = svg_obj.decode("utf-8")
    elif isinstance(svg_obj, str):                # Already a string
        svg = svg_obj
    else:
        raise TypeError(f"Unexpected SVG type: {type(svg_obj)}")

    # --- Draw rectangles on top of SVG ---
    cell_w, cell_h = SUB_IMG_SIZE
    levels = ["B", "C", "D", "E"]   # innermost B, outermost E
    base_margin = 2
    margin_step = 3

    rect_elems: List[str] = []

    # Substrate dotted rectangle in cell (0,0)
    rect_elems.append(
        f'<rect x="1" y="1" '
        f'width="{cell_w - 2}" height="{cell_h - 2}" '
        f'style="fill:none;stroke:black;stroke-width:1;stroke-dasharray:6,4" />'
    )

    # Map complete levels to mechanistic levels
    mech_for_comp = {"B": "Bm", "C": "Cm", "D": "Dm", "E": "Em"}

    # Prediction cells: nested rectangles for B/C/D/E
    # A level is "on" if either the complete or mechanistic column has a hit.
    for idx, row in enumerate(df_panel.itertuples(index=False), start=1):
        row_idx = idx // MOLS_PER_ROW
        col_idx = idx % MOLS_PER_ROW
        x0 = col_idx * cell_w
        y0 = row_idx * cell_h

        hit_levels: List[str] = []

        for lvl in levels:
            # Complete level column (B_match, C_match, ...)
            comp_col = f"{lvl}_match"
            comp_val = getattr(row, comp_col)
            comp_hit = False
            if isinstance(comp_val, str):
                v = comp_val.strip()
                if v and v not in NON_HITS:
                    comp_hit = True

            # Corresponding mechanistic level column (Bm_match_id, ...)
            mech_lvl = mech_for_comp[lvl]
            mech_col = f"{mech_lvl}_match_id"
            mech_val = getattr(row, mech_col)
            mech_hit = False
            if isinstance(mech_val, str):
                v2 = mech_val.strip()
                if v2 and v2 not in NON_HITS:
                    mech_hit = True

            if comp_hit or mech_hit:
                hit_levels.append(lvl)

        if not hit_levels:
            continue

        # Draw outermost E → innermost B for levels that actually hit
        for lvl in reversed(levels):
            if lvl not in hit_levels:
                continue

            order_index = levels.index(lvl)      # 0=B, 1=C, 2=D, 3=E
            margin = base_margin + (len(levels) - 1 - order_index) * margin_step

            rect_elems.append(
                f'<rect x="{x0 + margin}" y="{y0 + margin}" '
                f'width="{cell_w - 1 - 2*margin}" '
                f'height="{cell_h - 1 - 2*margin}" '
                f'style="fill:none;stroke:black;stroke-width:1" />'
            )

    # Inject rectangles before closing </svg>
    lower_svg = svg.lower()
    idx_close = lower_svg.rfind("</svg>")
    if idx_close == -1:
        raise RuntimeError("Could not find </svg> in RDKit SVG output")

    overlay = "\n".join(rect_elems) + "\n"
    svg_out = svg[:idx_close] + overlay + svg[idx_close:]

    # Save
    safe = sub_name.replace(" ", "_").replace("/", "_")
    svg_path = os.path.join("evodex_structure_panels", f"structures_{safe}.svg")
    with open(svg_path, "w") as f:
        f.write(svg_out)

    return svg_path


# Generate all panels
panel_paths = []
for sub_name, df in tables_by_substrate.items():
    path = make_structure_panel_svg(sub_name, df)
    panel_paths.append(path)

panel_paths[:3]

['evodex_structure_panels/structures_propanol.svg',
 'evodex_structure_panels/structures_ethyl_acetate.svg',
 'evodex_structure_panels/structures_propene.svg']

## 10. Download outputs

Download:

- the combined prediction + operator ID CSV
- the multi-page PDF of structure panels
- the operator metadata CSV

In [45]:
# === Build a PDF from the SVG panels ===

import os
import cairosvg
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO

svg_dir = "evodex_structure_panels"
pdf_path = "evodex_structure_panels.pdf"

panel_svgs = [
    os.path.join(svg_dir, f)
    for f in sorted(os.listdir(svg_dir))
    if f.endswith(".svg")
]

if not panel_svgs:
    raise FileNotFoundError("No SVG panels found in evodex_structure_panels/")

with PdfPages(pdf_path) as pdf:
    for svg_file in panel_svgs:
        # Convert SVG → PNG bytes using CairoSVG (vector-accurate)
        png_bytes = cairosvg.svg2png(url=svg_file, dpi=300)
        img = Image.open(BytesIO(png_bytes))

        # Put into PDF page
        fig, ax = plt.subplots(figsize=(8.5, 11))
        ax.imshow(img)
        ax.axis("off")
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

print("PDF built:", pdf_path)

PDF built: evodex_structure_panels.pdf


In [46]:
from google.colab import files

files.download("evodex_transformer_evodex_evaluation_table.csv")
files.download("evodex_structure_panels.pdf")
files.download("evodex_operator_hits_metadata.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Summary of EVODEX Matches per Substrate

The table below shows, for each substrate, how many of the model’s predictions
matched operators at each EVODEX abstraction level.

- **Bm/Cm/Dm/Em** columns count hits against mechanistic operators.  
- **B/C/D/E** columns count hits against complete operators.  

A value indicates how many of the generated products (out of 50) matched at that
level, giving a quick sense of which abstractions the model aligns with for each
substrate.

In [47]:
summary_rows = []
for sub, df_sub in full_df.groupby("substrate_name"):
    row = {"substrate_name": sub}
    for lvl in MECH_LEVELS:
        col = f"{lvl}_match_id"
        row[f"{lvl}_n"] = (df_sub[col].notna() & ~df_sub[col].isin(NON_HITS)).sum()
    for lvl in COMP_LEVELS:
        col = f"{lvl}_match"
        row[f"{lvl}_n"] = (df_sub[col].notna() & ~df_sub[col].isin(NON_HITS)).sum()
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

,substrate_name,Bm_n,Cm_n,Dm_n,Em_n,B_n,C_n,D_n,E_n
0,ethyl acetate,13,6,1,0,0,0,0,0
1,propanol,8,9,9,9,0,0,0,0
2,propene,0,0,0,0,0,0,0,0
